In [1]:
using HDF5
using PyPlot
using ProgressBars
using JLD2

In [87]:
function load_imgs(filenames, pathdir)
    imgs_atoms, imgs_bkg, imgs_dark = [], [], []

    for filename in filenames
        filename = pathdir*filename
        h5open(filename, "r") do file
            img_atoms = convert(Matrix{Int}, read(file["images/Vertical_Axis_Camera/in_situ_absorption/atoms"]))
            img_bkg = convert(Matrix{Int}, read(file["images/Vertical_Axis_Camera/in_situ_absorption/background"]))
            img_dark = convert(Matrix{Int}, read(file["images/Vertical_Axis_Camera/in_situ_absorption/dark"]))
            push!(imgs_atoms, img_atoms), push!(imgs_bkg, img_bkg), push!(imgs_dark, img_dark)
        end
    end
    return imgs_atoms, imgs_bkg, imgs_dark
end

function crop_imgs(imgs, x_crop, y_crop, mask)
    imgs_crop = []
    for i = 1:length(imgs)
        img_crop = imgs[i][y_crop, x_crop]
        push!(imgs_crop, img_crop .* mask)
    end
    return imgs_crop
end

function compute_OD(imgs_atoms, imgs_bkg, imgs_dark, mask)
    ODs = []
    for i in 1:length(imgs_atoms)
        OD = zeros(size(imgs_atoms_crop[i]))
        for a in length(imgs_atoms_crop[i])
            for b in length(imgs_atoms_crop[i][1])
                if (imgs_atoms_crop[i][a][b]+imgs_bkg_crop[i][a][b]+imgs_dark_crop[i][a][b] != 0) && ((imgs_atoms_crop[i][a][b] .- imgs_dark_crop[i][a][b]) ./ (imgs_bkg_crop[i][a][b] .- imgs_dark_crop[i][a][b]) > 0)
                    OD[a][b] = -log10((imgs_atoms_crop[i] .- imgs_dark_crop[i]) ./ (imgs_bkg_crop[i] .- imgs_dark_crop[i]))
                end
            end
        end
        # OD .= -log10.(OD)
        # OD[OD == NaN] .= 0
        push!(ODs, OD)
    end
    return ODs
end

function mask_ellipse(xc, yc, a, b, θ, x_crop, y_crop)
    mask = zeros(Integer, (length(y_crop), length(x_crop)))
    X = range(0, length(y_crop)-1) .- yc
    Y = range(0, length(x_crop)-1) .- xc
    for (i, x) in enumerate(X)
        for (j, y) in enumerate(Y)
            # Rotate back
            x_hor = x*sin(θ) + y*cos(θ)
            y_hor = x*cos(θ) - y*sin(θ)
            if (x_hor/a)^2 + (y_hor/b)^2 ≤ 1
                mask[i, j] = 1
            end
        end
    end
    return mask
end

mask_ellipse (generic function with 1 method)

In [76]:
pathdir_date = "data/04_07_25/"
dir_names_datasets = readdir(pathdir_date)
dir_names_datasets = dir_names_datasets[dir_names_datasets .!= ".DS_Store"]
x_crop = [1290:1560;]
y_crop = [1840:2210;]
xc, yc, a, b, θ = 140, 200, 70, 40, 1.1;

# Import and crop the images

In [92]:
imgs_atoms_crop_datasets, imgs_bkg_crop_datasets, imgs_dark_crop_datasets = [], [], []
mask = mask_ellipse(xc, yc, a, b, θ, x_crop, y_crop)

files_path = readdir(pathdir_date*dir_names_datasets[3])[1:2]
imgs_atoms, imgs_bkg, imgs_dark = load_imgs(files_path, pathdir_date*dir_names_datasets[3]*"/")
imgs_atoms_crop, imgs_bkg_crop, imgs_dark_crop = crop_imgs(imgs_atoms, x_crop, y_crop, mask), crop_imgs(imgs_bkg, x_crop, y_crop, mask), crop_imgs(imgs_dark, x_crop, y_crop, mask)


(Any[[0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0], [0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0]], Any[[0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0], [0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0]], Any[[0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0], [0 0 … 0 0; 0 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0]])

In [93]:
OD =compute_OD(imgs_atoms_crop, imgs_bkg_crop, imgs_dark_crop, mask)[1]


371×271 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

In [94]:
close("all")
fig, axs = subplots()
axs.imshow(compute_OD(imgs_atoms_crop, imgs_bkg_crop, imgs_dark_crop, mask)[1])

pygui(true); show()

In [ ]:
imgs_atoms_crop_datasets, imgs_bkg_crop_datasets, imgs_dark_crop_datasets = [], [], []
mask = mask_ellipse(xc, yc, a, b, θ, x_crop, y_crop)

for dir_name_datasets in ProgressBar(dir_names_datasets)
    files_path = readdir(pathdir_date*dir_name_datasets)
    imgs_atoms, imgs_bkg, imgs_dark = load_imgs(files_path, pathdir_date*dir_name_datasets*"/")
    imgs_atoms_crop, imgs_bkg_crop, imgs_dark_crop = crop_imgs(imgs_atoms, x_crop, y_crop, mask), crop_imgs(imgs_bkg, x_crop, y_crop, mask), crop_imgs(imgs_dark, x_crop, y_crop, mask)
    push!(imgs_atoms_crop_datasets, imgs_atoms_crop), push!(imgs_bkg_crop_datasets, imgs_bkg_crop), push!(imgs_dark_crop_datasets, imgs_dark_crop)
end

0.0%┣                                               ┫ 0/3 [00:07<00:-20, -7s/it]
33.3%┣██████████████▍                            ┫ 1/3 [00:55<Inf:Inf, InfGs/it]
66.7%┣██████████████████████████████▊               ┫ 2/3 [01:42<01:42, 102s/it]
100.0%┣██████████████████████████████████████████████┫ 3/3 [02:25<00:00, 72s/it]
100.0%┣██████████████████████████████████████████████┫ 3/3 [02:25<00:00, 72s/it]


In [5]:
@save "Imgs_crop_mask.jld2" imgs_atoms_crop_datasets imgs_bkg_crop_datasets imgs_dark_crop_datasets

In [6]:
@load "Imgs_crop.jld2" imgs_atoms_crop_datasets imgs_bkg_crop_datasets imgs_dark_crop_datasets

3-element Vector{Symbol}:
 :imgs_atoms_crop_datasets
 :imgs_bkg_crop_datasets
 :imgs_dark_crop_datasets

# Compute the ODs

In [22]:
files_path = readdir(pathdir_date*dir_names_datasets[1])
imgs_atoms, imgs_bkg, imgs_dark = load_imgs(files_path, pathdir_date*dir_names_datasets[1]*"/")

(Any[[197 201 … 218 210; 204 205 … 201 204; … ; 206 206 … 225 234; 202 199 … 209 216], [205 200 … 199 215; 200 202 … 199 202; … ; 195 202 … 197 210; 203 210 … 207 226], [202 209 … 222 200; 197 200 … 197 209; … ; 201 204 … 193 230; 200 199 … 210 223], [199 210 … 212 208; 198 200 … 199 225; … ; 209 198 … 199 231; 201 202 … 222 212], [200 198 … 200 199; 201 198 … 202 217; … ; 205 200 … 206 207; 209 204 … 197 205], [197 198 … 198 201; 201 195 … 209 205; … ; 202 201 … 224 217; 211 204 … 201 213], [202 194 … 207 213; 201 201 … 196 197; … ; 209 206 … 204 231; 204 202 … 200 207], [218 190 … 210 195; 197 209 … 199 207; … ; 204 210 … 201 229; 213 214 … 220 215], [197 208 … 213 199; 202 210 … 198 205; … ; 211 205 … 199 218; 205 207 … 213 225], [197 206 … 211 208; 197 197 … 208 204; … ; 211 198 … 197 244; 195 202 … 221 209]  …  [210 199 … 219 194; 199 200 … 198 208; … ; 200 195 … 197 203; 202 202 … 199 220], [202 192 … 202 199; 204 199 … 207 218; … ; 205 203 … 200 211; 200 210 … 198 221], [202 205

In [7]:
ODs_datasets = []
for i in ProgressBar(1:length(imgs_atoms_crop_datasets))
    push!(ODs_datasets, compute_OD(imgs_atoms_crop_datasets[i], imgs_bkg_crop_datasets[i], imgs_dark_crop_datasets[i]))
end

0.0%┣                                                ┫ 0/3 [00:00<00:00, -0s/it]
33.3%┣██████████████▍                            ┫ 1/3 [00:07<Inf:Inf, InfGs/it]
66.7%┣████████████████████████████████                ┫ 2/3 [00:09<00:09, 9s/it]
100.0%┣███████████████████████████████████████████████┫ 3/3 [00:10<00:00, 5s/it]
100.0%┣███████████████████████████████████████████████┫ 3/3 [00:10<00:00, 5s/it]


In [8]:
@save "ODs_crop_mask.jld2" ODs_datasets

In [9]:
@load "ODs_crop_mask.jld2" ODs_datasets

1-element Vector{Symbol}:
 :ODs_datasets

# Save the ODs images

In [10]:
if !isdir("imgs")
    mkdir("imgs")
end
if !isdir("imgs/ODs_mask")
    mkdir("imgs/ODs_mask")
end
if !isdir("imgs/ODs_mask/"*split(pathdir_date, "/")[2])
    mkdir("imgs/ODs_mask/"*split(pathdir_date, "/")[2])
end
if !isdir("imgs/neg_ODs_mask")
    mkdir("imgs/neg_ODs_mask")
end
if !isdir("imgs/neg_ODs_mask/"*split(pathdir_date, "/")[2])
    mkdir("imgs/neg_ODs_mask/"*split(pathdir_date, "/")[2])
end

### ODs

In [12]:
ODs_datasets[1][1]

701×601 Matrix{Float64}:
  0.0918059   -0.0        -0.0635381   …  -0.234083    0.0606978
 -0.0213583   -0.0        -0.0233786       0.0713331   0.123315
 -0.0933059   -0.0664904  -0.0670619       0.0460376   0.0544057
 -0.0837122   -0.0104319   0.0552872       0.0109164   0.0724917
 -0.057049     0.0458318   0.0879552       0.0942699  -0.0263289
  0.0589475    0.068936    0.00774394  …   0.224774   -0.0763624
  0.169751    -0.160195    0.0680533       0.0743824   0.0171889
 -0.00762896  -0.142703   -0.0137883      -0.0111364   0.0562772
 -0.0994466   -0.0070049   0.0659712       0.0942006  -0.0699651
 -0.146387    -0.0246225   0.0565735       0.0892791  -0.00582954
 -0.174072     0.0804158  -0.00406157  …  -0.0393234  -0.0619748
 -0.127664    -0.093628   -0.00711974      0.0420038   0.132947
  0.0220992    0.0480281  -0.0466212      -0.155766    0.0602959
  ⋮                                    ⋱               ⋮
  0.0205047    0.0191632   0.0821868       0.0425966  -0.20901
  0.0939855

In [11]:
close("all")
fig, axs = subplots()

for (i, ODs) in ProgressBar(enumerate(ODs_datasets))
    if !isdir("imgs/ODs_mask/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i])
        mkdir("imgs/ODs_mask/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i])
    end
    for (j, OD) in enumerate(ODs)
        img = axs.imshow(OD, cmap="plasma") #, aspect="auto"
        cb = colorbar(img)
        savefig("imgs/ODs_mask/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i]*"/$(j-1).png")
        cb.remove()
        axs.clear()
    end
end
close("all")

0.0%┣                                                ┫ 0/3 [00:00<00:00, -0s/it]
33.3%┣██████████████▍                            ┫ 1/3 [00:26<Inf:Inf, InfGs/it]


LoadError: InterruptException:

### Negative ODs

In [ ]:
close("all")
fig, axs = subplots()

for (i, ODs) in ProgressBar(enumerate(ODs_datasets))
    if !isdir("imgs/neg_ODs_mask/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i])
        mkdir("imgs/neg_ODs_mask/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i])
    end
    for (j, OD) in enumerate(ODs)
        img = axs.imshow(OD, cmap="bone", vmin=-0.2, vmax=0) #, aspect="auto"
        cb = colorbar(img)
        savefig("imgs/neg_ODs_mask/"*split(pathdir_date, "/")[2]*"/"*dir_names_datasets[i]*"/$(j-1).png")
        cb.remove()
        axs.clear()
    end
end
close("all")